# 04. Compare test on synthetic

This notebook runs the full cross-validation benchmark on the synthetic tumor data.
It implements the exact methodology from the Apollo paper: Feature Selection (Top 20) + SVM.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
import joblib

# Add project root to path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.loader import GliomaDataLoader
from src.models.baselines import ApolloPipeline

In [2]:
# --- Configuration ---
DATA_PATH = "../data/processed/glioma_dataset.h5"
SPLIT_FILE = "../data/splits/manual_splits.json"
SYNTHETIC_OUT_DIR = "../data/synthetic"

if not os.path.exists(SPLIT_FILE):
    print(f"❌ ERROR: Splits file not found at {SPLIT_FILE}")

print("Initializing Loader...")
# Ensure tumor_only=True so we only train on the data identified by our Spatio-Spectral Annotator
loader = GliomaDataLoader(
    data_path=DATA_PATH, 
    split_file=SPLIT_FILE, 
    memory_mapped=False, 
    tumor_only=True
)

print(f"✅ Loader initialized. Total Tumor Spectra: {loader.X.shape[0]}")

Initializing Loader...
📂 Loading data from ..\data\processed\glioma_dataset.h5...
🔬 'tumor_only=True': Filtering out healthy tissue...
   ✅ Filtered to 88487 Pure Tumor Spectra.
✅ Loader initialized. Total Tumor Spectra: 88487


## Run Benchmark (5-Fold CV)

In [8]:
import time
import os
import gc
import joblib
import numpy as np
import tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from src.models.cnn import build_1d_cnn 

# --- Setup Output Directories ---
MODEL_DIR = "../results/models/baselines"
os.makedirs(MODEL_DIR, exist_ok=True)

benchmark_results = {
    'Apollo': [],
    'RF_Standard': [],
    'XGBoost': [],
    '1D_CNN': []
}

print("🚀 Starting Memory-Optimized Multi-Model TRTR Benchmark...")

for fold_data in loader:
    fold_name = fold_data['fold_name']
    print(f"\n{'='*40}")
    print(f"🔄 Processing {fold_name}")
    print(f"{'='*40}")
    
    # 1. Load the synthetic data!
    X_train = np.load(f"{SYNTHETIC_OUT_DIR}/{fold_name}_X_synth.npy").astype(np.float32)
    y_train = np.load(f"{SYNTHETIC_OUT_DIR}/{fold_name}_y_synth.npy")
    X_val = fold_data['val'][0]
    y_val = fold_data['val'][1]
    
    # Fix 1-indexed labels if necessary
    if np.min(y_train) == 1:
        y_train = y_train - 1
        y_val = y_val - 1
        
    # --- MODEL 1: Apollo Baseline ---
    print("Training Apollo Pipeline...")
    apollo_pipeline = ApolloPipeline(n_features=20, random_state=42)

    # (Assuming apollo_pipeline is defined in a cell above this loop)
    apollo_pipeline.fit(X_train, y_train)
    y_pred_apollo = apollo_pipeline.predict(X_val)
    benchmark_results['Apollo'].append(accuracy_score(y_val, y_pred_apollo))
    
    # 💾 Save and Flush
    joblib.dump(apollo_pipeline, f"{MODEL_DIR}/apollo_{fold_name}.joblib")
    del y_pred_apollo # Keep pipeline in memory if defined globally, but delete preds
    gc.collect()
    
    # --- MODEL 2: Standard Random Forest ---
    print("Training Standard Random Forest...")
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred_rf = rf.predict(X_val)
    benchmark_results['RF_Standard'].append(accuracy_score(y_val, y_pred_rf))
    
    # 💾 Save and Flush
    joblib.dump(rf, f"{MODEL_DIR}/rf_{fold_name}.joblib")
    del rf, y_pred_rf
    gc.collect()
    
    # --- MODEL 3: XGBoost ---
    print("Training XGBoost...")
    # tree_method='hist' is faster and more memory efficient for large datasets
    xgb = XGBClassifier(objective='multi:softprob', num_class=6, random_state=42, n_jobs=-1, tree_method='hist')
    xgb.fit(X_train, y_train)
    y_pred_xgb = xgb.predict(X_val)
    benchmark_results['XGBoost'].append(accuracy_score(y_val, y_pred_xgb))
    
    # 💾 Save and Flush (XGBoost prefers saving as .json instead of joblib)
    xgb.save_model(f"{MODEL_DIR}/xgb_{fold_name}.json")
    del xgb, y_pred_xgb
    gc.collect()
    
    # --- MODEL 4: 1D-CNN ---
    print("Training 1D-CNN...")
    tf.keras.backend.clear_session()
    
    X_train_cnn = X_train.reshape(-1, 1738, 1)
    X_val_cnn = X_val.reshape(-1, 1738, 1)
    y_train_cnn = tf.keras.utils.to_categorical(y_train, num_classes=6)
    y_val_cnn = tf.keras.utils.to_categorical(y_val, num_classes=6)
    
    cnn_model = build_1d_cnn(input_shape=(1738, 1), num_classes=6)
    cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    
    cnn_model.fit(X_train_cnn, y_train_cnn, 
                  validation_data=(X_val_cnn, y_val_cnn),
                  epochs=30, batch_size=256, 
                  callbacks=[early_stop], verbose=0)
                  
    y_pred_cnn_probs = cnn_model.predict(X_val_cnn, verbose=0)
    y_pred_cnn = np.argmax(y_pred_cnn_probs, axis=1)
    benchmark_results['1D_CNN'].append(accuracy_score(y_val, y_pred_cnn))
    
    # 💾 Save and Flush
    cnn_model.save(f"{MODEL_DIR}/cnn_{fold_name}.keras")
    del cnn_model, X_train_cnn, X_val_cnn, y_train_cnn, y_val_cnn, y_pred_cnn_probs, y_pred_cnn
    tf.keras.backend.clear_session()
    gc.collect()

# --- Print Final Summary ---
print("\n🎉 === TRTR BENCHMARK COMPLETE ===")
for model_name, accuracies in benchmark_results.items():
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    print(f"{model_name:>15}: {mean_acc:.4f} ± {std_acc:.4f}")

🚀 Starting Memory-Optimized Multi-Model TRTR Benchmark...

🔄 Processing fold_0
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (90000, 1738, 1) features...


ValueError: Found array with dim 3, while dim <= 2 is required by RandomForestClassifier.

In [3]:
import time
import os
import gc
import joblib
import numpy as np
import tensorflow as tf
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from src.models.baselines import ApolloPipeline
from src.models.cnn import build_1d_cnn 

# --- Setup Output Directories ---
MODEL_DIR = "../results/models/baselines"
os.makedirs(MODEL_DIR, exist_ok=True)

benchmark_results = {
    'Apollo': [],
    'RF_Standard': [],
    'XGBoost': [],
    '1D_CNN': []
}

print("🚀 Starting Memory-Optimized Multi-Model TRTR Benchmark...")

for fold_data in loader:
    fold_name = fold_data['fold_name']
    print(f"\n{'='*40}")
    print(f"🔄 Processing {fold_name}")
    print(f"{'='*40}")
    
    # 1. Load the synthetic data
    X_train = np.load(f"{SYNTHETIC_OUT_DIR}/{fold_name}_X_synth.npy").astype(np.float32)
    y_train = np.load(f"{SYNTHETIC_OUT_DIR}/{fold_name}_y_synth.npy")
    X_val = fold_data['val'][0]
    y_val = fold_data['val'][1]

    # Fix 1-indexed labels if necessary
    if np.min(y_train) == 1:
        y_train = y_train - 1
        y_val = y_val - 1

    # -------------------------------------------------
    # 🔹 Ensure 2D input for classical ML models
    # -------------------------------------------------
    if X_train.ndim == 3:
        X_train_tab = X_train.reshape(X_train.shape[0], -1)
        X_val_tab = X_val.reshape(X_val.shape[0], -1)
    else:
        X_train_tab = X_train
        X_val_tab = X_val

    # --- MODEL 1: Apollo Baseline ---
    print("Training Apollo Pipeline...")
    apollo_pipeline = ApolloPipeline(n_features=20, random_state=42)
    apollo_pipeline.fit(X_train_tab, y_train)
    y_pred_apollo = apollo_pipeline.predict(X_val_tab)
    benchmark_results['Apollo'].append(accuracy_score(y_val, y_pred_apollo))

    joblib.dump(apollo_pipeline, f"{MODEL_DIR}/apollo_{fold_name}.joblib")
    del y_pred_apollo
    gc.collect()

    # --- MODEL 2: Standard Random Forest ---
    print("Training Standard Random Forest...")
    rf = RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train_tab, y_train)
    y_pred_rf = rf.predict(X_val_tab)
    benchmark_results['RF_Standard'].append(accuracy_score(y_val, y_pred_rf))

    joblib.dump(rf, f"{MODEL_DIR}/rf_{fold_name}.joblib")
    del rf, y_pred_rf
    gc.collect()

    # --- MODEL 3: XGBoost ---
    print("Training XGBoost...")
    xgb = XGBClassifier(
        objective='multi:softprob',
        num_class=6,
        random_state=42,
        n_jobs=-1,
        tree_method='hist'
    )
    xgb.fit(X_train_tab, y_train)
    y_pred_xgb = xgb.predict(X_val_tab)
    benchmark_results['XGBoost'].append(accuracy_score(y_val, y_pred_xgb))

    xgb.save_model(f"{MODEL_DIR}/xgb_{fold_name}.json")
    del xgb, y_pred_xgb
    gc.collect()

    # --- MODEL 4: 1D-CNN ---
    print("Training 1D-CNN...")
    tf.keras.backend.clear_session()

    # Ensure proper CNN shape (N, 1738, 1)
    if X_train.ndim == 2:
        X_train_cnn = X_train.reshape(-1, 1738, 1)
        X_val_cnn = X_val.reshape(-1, 1738, 1)
    else:
        X_train_cnn = X_train
        X_val_cnn = X_val

    y_train_cnn = tf.keras.utils.to_categorical(y_train, num_classes=6)
    y_val_cnn = tf.keras.utils.to_categorical(y_val, num_classes=6)

    cnn_model = build_1d_cnn(input_shape=(1738, 1), num_classes=6)
    cnn_model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )

    cnn_model.fit(
        X_train_cnn,
        y_train_cnn,
        validation_data=(X_val_cnn, y_val_cnn),
        epochs=30,
        batch_size=256,
        callbacks=[early_stop],
        verbose=0
    )

    y_pred_cnn_probs = cnn_model.predict(X_val_cnn, verbose=0)
    y_pred_cnn = np.argmax(y_pred_cnn_probs, axis=1)
    benchmark_results['1D_CNN'].append(accuracy_score(y_val, y_pred_cnn))

    cnn_model.save(f"{MODEL_DIR}/cnn_{fold_name}.keras")
    del cnn_model, X_train_cnn, X_val_cnn, y_train_cnn, y_val_cnn, y_pred_cnn_probs, y_pred_cnn
    tf.keras.backend.clear_session()
    gc.collect()


# --- Print Final Summary ---
print("\n🎉 === TRTR BENCHMARK COMPLETE ===")
for model_name, accuracies in benchmark_results.items():
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    print(f"{model_name:>15}: {mean_acc:.4f} ± {std_acc:.4f}")

🚀 Starting Memory-Optimized Multi-Model TRTR Benchmark...

🔄 Processing fold_0
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (90000, 1738) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
Training Standard Random Forest...
Training XGBoost...
Training 1D-CNN...


🔄 Processing fold_1
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (90000, 1738) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
Training Standard Random Forest...
Training XGBoost...
Training 1D-CNN...

🔄 Processing fold_2
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (90000, 1738) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
Training Standard Random Forest...
Training XGBoost...
Training 1D-CNN...

🔄 Processing fold_3
Training Apollo Pipeline...
  [Apollo] Fitting RF Selector on (90000, 1738) features...
  [Apollo] Selected 20 features.
  [Apollo] Fitting SVM Classifier...
Tra

KeyboardInterrupt: 

In [6]:
print("\n🎉 === TRTR BENCHMARK COMPLETE ===")
for model_name, accuracies in benchmark_results.items():
    mean_acc = np.mean(accuracies)
    std_acc = np.std(accuracies)
    print(f"{model_name:>15}: {mean_acc:.4f} ± {std_acc:.4f}")


🎉 === TRTR BENCHMARK COMPLETE ===
         Apollo: 0.4148 ± 0.0850
    RF_Standard: 0.4385 ± 0.1126
        XGBoost: 0.4706 ± 0.1014
         1D_CNN: 0.2111 ± 0.0881
